[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

/Users/renyumeng/文稿/1.code/TorchCode/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)
        
        self.head_dim = d_model // num_heads
        self.num_heads = num_heads
          # Initialize W_q, W_k, W_v, W_o

    def forward(self, Q:torch.Tensor, K:torch.Tensor, V:torch.Tensor):
        # shape (batchsize,se_q,d_model)
        batchsize,se_q,d_q = Q.shape
        batchsize,se_k,d_k = K.shape
        batchsize,se_v,d_v = V.shape
        
        # shape (batchsize,num_heads,seq_len,head_dim)
        q:torch.Tensor = self.W_q(Q).reshape(batchsize,-1,self.num_heads,self.head_dim).transpose(1,2)
        k:torch.Tensor = self.W_k(K).reshape(batchsize,-1,self.num_heads,self.head_dim).transpose(1,2)
        v:torch.Tensor = self.W_v(V).reshape(batchsize,-1,self.num_heads,self.head_dim).transpose(1,2)
        # shape (batchsize,num_heads,se_q,se_k)
        score = q@k.transpose(-2,-1)
        score = torch.softmax(score / math.sqrt(self.head_dim),dim=-1)
        # shape (batchsize,num_heads,se_q,head_dim)
        content = score @ v
        content = content.transpose(1,2).reshape(batchsize,-1,d_q)
        return self.W_o(content)
        

In [5]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [6]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (8.3ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.5ms)
  ✅ [3/6] Numerical correctness vs reference (14.2ms)
  ✅ [4/6] Gradient flow (50.0ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (0.7ms)
  ✅ [6/6] Different heads give different outputs (3.7ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (77.5ms total)
  Progress saved. Run status() to see your dashboard.

